In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

In [2]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
# =========================
# 1) Load data
# =========================
DATA_PATH = "/content/CustChurnTestDatasetDemo.csv"  # <- Colab upload location
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (500, 12)
Columns: ['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction', 'Churn']


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0


In [4]:
# =========================
# 2) Choose target column
# =========================

TARGET_COL = None

if TARGET_COL is None:
    candidate_names = ["churn", "churn_type", "churntype", "label", "target", "class", "y"]
    lower_map = {c.lower(): c for c in df.columns}
    for key in candidate_names:
        for lc, orig in lower_map.items():
            if key == lc or key in lc:
                TARGET_COL = orig
                break
        if TARGET_COL is not None:
            break

if TARGET_COL is None:
    TARGET_COL = df.columns[-1]  # fallback

print("Using TARGET_COL =", TARGET_COL)

X = df.drop(columns=[TARGET_COL])
y_raw = df[TARGET_COL].astype(str)

Using TARGET_COL = Churn


In [5]:
# =========================
# 3) Train/test split
# =========================
X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X, y_raw, test_size=0.2, random_state=SEED, stratify=y_raw
)

In [6]:
# =========================
# 4) Preprocessing
# =========================
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ]
)

X_train_pp = preprocess.fit_transform(X_train)
X_test_pp = preprocess.transform(X_test)

# Dense arrays for Keras
if hasattr(X_train_pp, "toarray"):
    X_train_pp = X_train_pp.toarray()
    X_test_pp = X_test_pp.toarray()

In [7]:
# =========================
# 5) Encode labels (binary vs multi-class)
# =========================
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

num_classes = len(le.classes_)
is_binary = (num_classes == 2)

print("Classes:", list(le.classes_))

Classes: ['0', '1']


The methhod build_model(...) creates and returns a neural network (ANN) model in Keras (TensorFlow’s high-level API).
It’s written as a function because we want to build many versions of the model during hyperparameter tuning (different number of layers, different neurons, different regularization, etc.). - Requirement for Assignment 2.

Imagine this model as a model factory that produces models :).

In this model factory (build_model), I pass in settings (hyperparameters) that constructs an ANN network with those settings, The model factory compiles the ANN network (sets how it learns), and returns the ANN network model.

**def build_model(input_dim, n_hidden_layers, units, l1, l2, lr=1e-3, dropout=0.2):**

Parameters to the method build_model:
*   input_dim: number of input features (columns) going into the network after preprocessing, e.g., scaling and one-hot encoding.
*   n_hidden_layer: how many hidden layers we want, e.g., 1, 2, 3, etc.
*   units: number of neurons in each hidden layer (e.g., 32, 64, 128).
*   l1, l2: regularization strengths (penalties) to reduce overfitting.
*   lr: learning rate for the optimizer (default 0.001). Controls how big each weight update step is.
*   dropout: dropout rate (default 0.2 = 20%). A technique to reduce overfitting.

**reg = regularizers.l1_l2(l1=l1, l2=l2)**

Regularization is controller that controls the network (ANN) so that it does not get too complicated. Two types of regularization rules - L1 and L2.


*   L1 regularization tends to push some weights to exactly 0 → can behave like feature selection (simplifies the model).
*   L2 regularization tends to make weights smaller (but not usually exactly 0) → keeps the model smooth and stable.

Keras will add these penalties to the loss function automatically.



L1 regularization tends to push some weights to exactly 0
→ can behave like feature selection (simplifies the model).

L2 regularization tends to make weights smaller (but not usually exactly 0)
→ keeps the model smooth and stable.

Keras will add these penalties to your loss function automatically.

**model = keras.Sequential()**

We specify how the ANN layers are organized (structured). The **Sequential()** method means the model's layers are stacked one after another, like: Input → Hidden → Hidden → Output

This is the simplest type of ANN architecture.

**model.add(layers.Input(shape=(input_dim,)))**

The add() method tells Keras:
“Each training example is a vector of length input_dim.”. E.g., if after preprocessing we have 50 columns/features,then input_dim = 50, and the input shape is (50,).

for _ in range(n_hidden_layers):

**model.add(layers.Dense(units, activation="relu", kernel_regularizer=reg))**

This is to loop through n_hidden_layers times to add density unit, the non-linearlity to learn complex pattern, and the regularizer.

*   Dense means “fully connected”: every neuron connects to every neuron in the previous layer.
*   units is how many neurons are in the layer.
*   ReLU activation (relu) is a very common choice for hidden layers. It introduces non-linearity (so the model can learn complex patterns), and it is efficient and usually trains well.
*   kernel_regularizer=reg applies L1/L2 penalties to the weights of this layer.


**layers.Dropout(dropout)**

The method Dropout() is a training trick. During each training step, it randomly “turns off” a fraction of neurons (e.g., 20%). This forces the network not to rely too heavily on any single neuron,which helps reduce overfitting.

Removing Dropout(...) will not make the model “fail” (it will still compile and train), but it can change accuracy—often in a very predictable way.

Without dropout, the network has more effective capacity every step, so it can fit the training data more easily. If the model was overfitting (training accuracy high, validation/test accuracy noticeably lower), dropout usually helps. Removing it can reduce test accuracy.

If the model was underfitting (both training and test accuracy low), dropout can make learning harder. Removing it can increase test accuracy.

If we have a large dataset and/or strong regularization (L2/L1) and not too large a network, dropout may not be necessary, and removing it may change little.

**Output layer and correct loss function (binary vs multi-class)**

This part depends on whether the churn type (target class) is binary (2 classes) or multi-class (3+ classes).

**if is_binary:**
    **model.add(layers.Dense(1, activation="sigmoid"))
    loss = "binary_crossentropy"**

If the target class is binary, we just need one output neuron, producing a value between 0 and 1.

The sigmoid converts the output to a probability-like number. E.g., 0.92 means “92% chance of churn type = class 1”.

We use Binary_crossentropy if the target class is binary. Binary_crossentropy is the correct loss for binary classification.

**else:**
    **model.add(layers.Dense(num_classes, activation="softmax"))
    loss = "sparse_categorical_crossentropy"**
otherwise (else), the target class is multi-class. The Output has num_classes neurons, one per class. Example: if churn has 3 types, you get 3 output neurons. The softmax method converts the outputs into probabilities that sum to 1. E.g., [0.10, 0.70, 0.20].

sparse_categorical_crossentropy is used when your labels are integers like: class 0, class 1, class 2
and not one-hot encoded labels.

# Compile the model

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr),
    loss=loss,
    metrics=["accuracy"]
)

Compiling is like setting the “training configuration” to the ANN model:

*   optimizer=Adam(lr): Adam is a popular optimizer that adapts learning rates per parameter. The learning rate (lr) controls step size.
*   loss=loss: loss measures “how wrong” the model is. Training will try to minimize loss.
*   metrics=["accuracy"]: accuracy is printed during training/evaluation. it is not what is minimized directly (loss is), but it is easy to interpret.

In [8]:
# =========================
# 6) Model builder (ANN in TF/Keras)
# =========================
def build_model(input_dim, n_hidden_layers, units, l1, l2, lr=1e-3, dropout=0.2):
    reg = regularizers.l1_l2(l1=l1, l2=l2)

    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for _ in range(n_hidden_layers):
        model.add(layers.Dense(units, activation="relu", kernel_regularizer=reg))
        model.add(layers.Dropout(dropout))

    # Output layer and correct loss function (binary vs multi-class)
    if is_binary:
        model.add(layers.Dense(1, activation="sigmoid"))
        loss = "binary_crossentropy"
    else:
        model.add(layers.Dense(num_classes, activation="softmax"))
        loss = "sparse_categorical_crossentropy"

    # Comppile the model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=loss,
        metrics=["accuracy"]
    )
    return model

In [9]:
# =========================
# 7) Hyperparameter grid (meets requirement)
# =========================
grid_layers = [1, 2, 3]          # (a) number of hidden layers (>=2 candidates)
grid_units  = [32, 64, 128]      # (b) neurons per layer (>=2 candidates)
grid_l1     = [0.0, 1e-4]        # (c) L1 regularization (>=2 candidates)
grid_l2     = [0.0, 1e-4]        # (c) L2 regularization (>=2 candidates)

EPOCHS = 100
BATCH_SIZE = 32

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

# validation split from training data for tuning
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_pp, y_train, test_size=0.2, random_state=SEED, stratify=y_train
)

best_val_acc = -1.0
best_params = None
best_model = None

input_dim = X_tr.shape[1]
results = []

for n_layers in grid_layers:
    for units in grid_units:
        for l1 in grid_l1:
            for l2 in grid_l2:
                tf.keras.backend.clear_session()

                model = build_model(input_dim, n_layers, units, l1, l2)
                hist = model.fit(
                    X_tr, y_tr,
                    validation_data=(X_val, y_val),
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    verbose=0,
                    callbacks=[early_stop]
                )

                val_acc = float(np.max(hist.history["val_accuracy"]))
                val_loss = float(np.min(hist.history["val_loss"]))
                results.append((n_layers, units, l1, l2, val_acc, val_loss))

                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    best_params = (n_layers, units, l1, l2)
                    best_model = model

# Show top configs
results_sorted = sorted(results, key=lambda x: (-x[4], x[5]))
print("\nTop 5 configs (layers, units, l1, l2, val_acc, val_loss):")
for r in results_sorted[:5]:
    print(r)

print("\nBest params:", best_params, "Best val_acc:", best_val_acc)


Top 5 configs (layers, units, l1, l2, val_acc, val_loss):
(3, 128, 0.0001, 0.0, 0.9750000238418579, 0.31419458985328674)
(2, 128, 0.0, 0.0, 0.9624999761581421, 0.12558788061141968)
(2, 64, 0.0, 0.0, 0.9624999761581421, 0.12746576964855194)
(3, 128, 0.0, 0.0001, 0.9624999761581421, 0.14118006825447083)
(2, 128, 0.0001, 0.0, 0.9624999761581421, 0.2259369194507599)

Best params: (3, 128, 0.0001, 0.0) Best val_acc: 0.9750000238418579


In [10]:
# =========================
# 8) Final training on full training set + evaluate on test (required)
# =========================
best_n_layers, best_units, best_l1, best_l2 = best_params

tf.keras.backend.clear_session()
final_model = build_model(X_train_pp.shape[1], best_n_layers, best_units, best_l1, best_l2)

final_model.fit(
    X_train_pp, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,
    callbacks=[early_stop]
)

train_loss, train_acc = final_model.evaluate(X_train_pp, y_train, verbose=0)
test_loss, test_acc = final_model.evaluate(X_test_pp, y_test, verbose=0)

print("\n=== Required Metrics ===")
print(f"Training Loss: {train_loss:.4f} | Training Accuracy: {train_acc:.4f}")
print(f"Test Loss:     {test_loss:.4f} | Test Accuracy:     {test_acc:.4f}")

# Predictions + extra diagnostics (good for report)
if is_binary:
    y_prob = final_model.predict(X_test_pp).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
else:
    y_prob = final_model.predict(X_test_pp)
    y_pred = np.argmax(y_prob, axis=1)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))


=== Required Metrics ===
Training Loss: 0.2918 | Training Accuracy: 0.9725
Test Loss:     0.4855 | Test Accuracy:     0.9200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step

Confusion Matrix:
[[71  4]
 [ 4 21]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95        75
           1       0.84      0.84      0.84        25

    accuracy                           0.92       100
   macro avg       0.89      0.89      0.89       100
weighted avg       0.92      0.92      0.92       100

